In [ ]:
# figure out the direction of H2O2 reactions to be removed
# using 's' > 0 is easy, but may not be the best method (we see reactions that are not supposed to happen)

In [3]:
import networkExpansionPy.folds as nf
import pandas as pd
from collections import Counter
from pathlib import PurePath, Path
import csv
import ast

asset_path = nf.asset_path

In [4]:
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']

NEW_METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.02Jun2024_peroxide_fix.pkl") # path to metabolism object pickle
OLD_METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # alt. metabolism model

old_metabolism = pd.read_pickle(OLD_METABOLISM_PATH)
new_metabolism = pd.read_pickle(NEW_METABOLISM_PATH)

print(len(old_metabolism.network), len(new_metabolism.network))


105986 105933


In [44]:
print(old_metabolism.network[old_metabolism.network.isin(['R12454']).any(axis=1)])

           rn direction     cid    s   ub            lb stoich_str rn_old
73614  R12454   forward  C21284 -1.0  0.1  1.000000e-07        NaN    NaN
73615  R12454   forward  C00027 -1.0  0.1  1.000000e-07        NaN    NaN
73616  R12454   forward  C00001  2.0  0.1  1.000000e-07        NaN    NaN
73617  R12454   forward  C22173  1.0  0.1  1.000000e-07        NaN    NaN
73618  R12454   forward  C00011  1.0  0.1  1.000000e-07        NaN    NaN
73619  R12454   reverse  C21284  1.0  0.1  1.000000e-07        NaN    NaN
73620  R12454   reverse  C00027  1.0  0.1  1.000000e-07        NaN    NaN
73621  R12454   reverse  C00001 -2.0  0.1  1.000000e-07        NaN    NaN
73622  R12454   reverse  C22173 -1.0  0.1  1.000000e-07        NaN    NaN
73623  R12454   reverse  C00011 -1.0  0.1  1.000000e-07        NaN    NaN


In [5]:
def csv2dict(csv_file_path):
    result_dict = {}
    with open(csv_file_path, 'r') as csv_file:
        csv_reader = csv.reader(csv_file)
        
        for row in csv_reader:
            try:
                result_dict[row[0]] = ast.literal_eval(row[1])  # value is list
            except:
                result_dict[row[0]] = row[1]
    return result_dict

rn2eqn = csv2dict('../data/assets/rn2eqn_SI.csv')

In [15]:
for rn in rn2eqn.keys():
    if rn in H2O2_rns:
        print(rn, rn2eqn[rn])

R00017_v1 2C00125 + 2C00001 + Z00025 => 2C00126 + C00027 + Z00025
R03532_v1 2C00001 + C00028 + Z00025 + 2C00001 + C00028 + Z00025 => C00030 + C00027 + Z00025 + C00030 + C00027 + Z00025
R09507_v1 C01335 + C00001 + Z00025 + Z00010 => C01371 + C00027 + Z00025 + Z00010
R09740_v1 C05102 + C00001 + Z00025 => C00162 + C00027 + Z00025
R09741_v1 C19861 + C00001 + Z00025 => C00162 + C00027 + Z00025
R11522 2C00011 + 4C00001 + C00032 => 2C00027 + C21284
R12454 2C00001 + C22173 + C00011 => C21284 + C00027
R12455 2C00001 + C00032 + C00011 => C22173 + C00027


In [ ]:
"""
R00017_v1 2C00125 + 2C00001 + Z00025 => 2C00126 + C00027 + Z00025
R03532_v1 2C00001 + C00028 + Z00025 + 2C00001 + C00028 + Z00025 => C00030 + C00027 + Z00025 + C00030 + C00027 + Z00025
R09507_v1 C01335 + C00001 + Z00025 + Z00010 => C01371 + C00027 + Z00025 + Z00010
R09740_v1 C05102 + C00001 + Z00025 => C00162 + C00027 + Z00025
R09741_v1 C19861 + C00001 + Z00025 => C00162 + C00027 + Z00025
R11522 2C00011 + 4C00001 + C00032 => 2C00027 + C21284
R12454 2C00001 + C22173 + C00011 => C21284 + C00027
R12455 2C00001 + C00032 + C00011 => C22173 + C00027
"""

# for all H2O2 reactions, it's the reverse reaction that should be removed

In [7]:
metabolism = pd.read_pickle(OLD_METABOLISM_PATH)

condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['s'] > 0) & (metabolism.network['cid'] == 'C00027'))
metabolism.network[condition]

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
65814,R11522,reverse,C00027,2.0,0.1,1.000000e-07,NaN,NaN
73620,R12454,reverse,C00027,1.0,0.1,1.000000e-07,NaN,NaN
73630,R12455,reverse,C00027,1.0,0.1,1.000000e-07,NaN,NaN
21233,R03532_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R03532
83,R00017_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R00017
21233,R03532_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R03532
54203,R09740_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R09740
54211,R09741_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R09741
52726,R09507_v1,reverse,C00027,1.0,0.1,1.000000e-07,NaN,R09507


In [9]:
condition = ((metabolism.network['rn'].isin(H2O2_rns)))
metabolism.network[condition].head(20)

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
65809,R11522,forward,C00027,-2.0,0.1,1.000000e-07,NaN,NaN
65810,R11522,forward,C21284,-1.0,0.1,1.000000e-07,NaN,NaN
65811,R11522,forward,C00011,2.0,0.1,1.000000e-07,NaN,NaN
65812,R11522,forward,C00001,4.0,0.1,1.000000e-07,NaN,NaN
65813,R11522,forward,C00032,1.0,0.1,1.000000e-07,NaN,NaN
65814,R11522,reverse,C00027,2.0,0.1,1.000000e-07,NaN,NaN
65815,R11522,reverse,C21284,1.0,0.1,1.000000e-07,NaN,NaN
65816,R11522,reverse,C00011,-2.0,0.1,1.000000e-07,NaN,NaN
65817,R11522,reverse,C00001,-4.0,0.1,1.000000e-07,NaN,NaN
65818,R11522,reverse,C00032,-1.0,0.1,1.000000e-07,NaN,NaN
